# Stage 3 — 全レース再スクレイプ（v2）

history.db の全レース（約4,900件）をJRA公式から再スクレイプし、
surface/race_class/track_condition/weather/斤量/性齢/体重/オッズ等を一括補完する。

**URLナビゲーション**: 一括取得v5と同じ `month_suffix_map.json` → `get_kaisai_for_month()` を使用

**保存方針**: 既存レースも全フィールドを上書き（INSERT OR IGNOREではなくUPDATE）

**推定所要時間**: 4,900レース × 約1.5秒 ≈ 2〜3時間

In [ ]:
# ── セル1: セットアップ ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, json, re, time, sqlite3, requests, pickle
from bs4 import BeautifulSoup
from collections import defaultdict, deque
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
HIST_DB  = f'{BASE_DIR}/data/history.db'
print(f'✅ セットアップ完了')

In [ ]:
# ── セル2: src/ 強制アップデート ──────────────────────────────
import urllib.request, sys
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
files = [
    'src/tools/__init__.py', 'src/tools/tune_weights.py',
    'src/tools/calibrate.py', 'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py', 'src/features/engine.py',
    'src/utils/config.py', 'src/utils/db.py', 'src/utils/model_registry.py',
    'src/scraper/parser.py', 'src/scraper/jra_scraper.py',
    'src/models/__init__.py', 'src/models/calibration.py', 'src/models/predict.py',
    'src/betting/__init__.py', 'src/betting/make_bets.py',
    'src/betting/ev_filter.py', 'src/betting/app_json.py',
]
for rel in files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}', dest)
    print(f'OK {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]
print('done')

In [ ]:
# ── セル3: history.db バックアップ ──────────────────────────
import shutil, datetime
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
bak = f'{BASE_DIR}/data/history_backup_{ts}.db'
shutil.copy2(HIST_DB, bak)
sz = os.path.getsize(bak) / 1024 / 1024
print(f'✅ バックアップ: {bak} ({sz:.1f} MB)')

In [ ]:
# ── セル4: 関数定義（v5ベース + 新フィールド拡張）────────────

HEADERS  = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
JRA_BASE = 'https://www.jra.go.jp'
PLACE_NAMES = {'01':'札幌','02':'函館','03':'福島','04':'新潟','05':'東京',
               '06':'中山','07':'中京','08':'京都','09':'阪神','10':'小倉'}

def calc_suffix(r01, r):
    if r <= 9:    return f'{(r01 + (r-1)*181) % 256:02X}'
    elif r == 10: return f'{(r01 + 8*181 + 245) % 256:02X}'
    else:         return f'{(r01 + 8*181 + 245 + (r-10)*181) % 256:02X}'

def find_r01_result(base, date, sess):
    for s in range(256):
        cn = f'{base}01{date}/{s:02X}'
        try:
            r = sess.post(f'{JRA_BASE}/JRADB/accessS.html',
                          data={'CNAME': cn}, timeout=3)
            r.encoding = 'shift_jis'
            if 'パラメータエラー' not in r.text and BeautifulSoup(r.text, 'lxml').find_all('table'):
                return s
        except Exception:
            pass
        time.sleep(0.05)
    return None

def get_kaisai_for_month(ym, suffix, sess):
    resp = sess.post(f'{JRA_BASE}/JRADB/accessS.html',
                     data={'CNAME': f'pw01skl10{ym}/{suffix}'}, timeout=15)
    resp.encoding = 'shift_jis'
    soup = BeautifulSoup(resp.text, 'lxml')
    links = {}
    for tag in soup.find_all(onclick=True):
        m = re.search(r'pw01srl10(\d{2})(\d{4})(\d{2})(\d{2})(\d{8})/(\w{2})', tag['onclick'])
        if m:
            pc, year, kai, nichi, date, sx = m.groups()
            sde_base = f'pw01sde10{pc}{year}{kai}{nichi}'
            if sde_base not in links:
                links[sde_base] = date
    return links

# ── ヘルパー関数（parse_result_page内で使用）──
def _parse_finish_time(text):
    if not text: return None
    t = str(text).strip().replace(' ', '')
    m = re.match(r'^(\d+)[:\.](\d{1,2})\.(\d)$', t)
    if m: return int(m.group(1))*60 + int(m.group(2)) + int(m.group(3))/10
    m = re.match(r'^(\d+(?:\.\d+)?)$', t)
    if m: return float(m.group(1))
    return None

def _parse_margin(text):
    if not text or text in ('---', '-', ''): return 0.0
    named = {'ハナ': 0.1, 'クビ': 0.2, 'アタマ': 0.3, '大差': 10.0}
    if text in named: return named[text]
    m = re.match(r'^(\d+)\s+(\d+)/(\d+)$', text.strip())
    if m: return int(m.group(1)) + int(m.group(2))/int(m.group(3))
    m = re.match(r'^(\d+)/(\d+)$', text.strip())
    if m: return int(m.group(1))/int(m.group(2))
    m = re.match(r'^(\d+(?:\.\d+)?)$', text.strip())
    if m: return float(m.group(1))
    return 0.0

def _extract_class(header):
    import unicodedata
    t = unicodedata.normalize('NFKC', header or '')
    if re.search(r'\(\s*G\s*3\s*\)|\(GIII\)', t): return 'G3'
    if re.search(r'\(\s*G\s*2\s*\)|\(GII\)',  t): return 'G2'
    if re.search(r'\(\s*G\s*1\s*\)|\(GI\)',   t): return 'G1'
    if re.search(r'\(\s*L\s*\)', t):              return 'L'
    if '3勝クラス' in t: return '3勝'
    if '2勝クラス' in t: return '2勝'
    if '1勝クラス' in t: return '1勝'
    if '未勝利' in t:   return '未勝利'
    if '新馬' in t:     return '新馬'
    if 'オープン' in t: return 'OP'
    return ''

def parse_result_page(soup, rc, race_num, date, pc):
    """v5ベース + 新フィールド（斤量/性齢/体重/オッズ/タイム/通過順）追加"""
    try:
        tables = soup.find_all('table')
        if not tables: return None
        header = tables[0].get_text(' ', strip=True)

        # 距離・コース
        dm = re.search(r'([\d,]+)\s*メートル\s*[（(]\s*([芝ダ])', header)
        distance = int(dm.group(1).replace(',', '')) if dm else 0
        surface  = '芝' if dm and dm.group(2) == '芝' else ('ダート' if dm else '不明')
        if surface == '不明':
            print(f'  ⚠ surface判定失敗: {header[:60]}')

        # 障害スキップ
        if any(kw in header for kw in ['障害', 'ジャンプ']): return None

        # レース名
        c = header.replace('本賞金','').replace('付加賞','')
        sp = re.search(r'([\u3040-\u9fff\u30A0-\u30FFa-zA-Z0-9]+(?:賞|杯|記念|特別|ステークス|カップ|トロフィー))', c)
        gen = re.search(r'(\d歳(?:以上)?(?:未勝利|1勝クラス|2勝クラス|3勝クラス|オープン))', header)
        if sp and sp.group(1) not in ('本賞','付加賞') and len(sp.group(1)) >= 3:
            rname = sp.group(1).strip()
        elif gen: rname = gen.group(1).strip()
        else:     rname = f'R{race_num:02d}'

        # 馬場状態・クラス・天候
        tc_m = re.search(r'(良|稍重|重|不良)', header)
        track_condition = tc_m.group(1) if tc_m else None
        race_class = _extract_class(header)
        wt_m = re.search(r'天候[\s:：]*(晴|曇|雨|小雨|雪)', header)
        weather = wt_m.group(1) if wt_m else None

        # ラップタイム
        lap_times = first_3f = last_3f = None
        for tbl in tables:
            txt = tbl.get_text(' ', strip=True)
            if 'ハロンタイム' in txt:
                m = re.search(r'ハロンタイム\s+([\d.\s\-]+)', txt)
                if m:
                    laps = [float(x) for x in re.findall(r'\d+\.\d', m.group(1).strip())]
                    if laps:
                        lap_times = '-'.join(str(x) for x in laps)
                        first_3f  = round(sum(laps[:3]), 1)
                        last_3f   = round(sum(laps[-3:]), 1)
                break

        # 出走馬パース
        finishers = []
        for row in tables[0].find_all('tr'):
            cells = row.find_all('td')
            if len(cells) < 10: continue
            tx = [c.get_text(' ', strip=True) for c in cells]
            pm = re.match(r'^(\d+)$', tx[0].strip())
            if not pm: continue
            place = int(pm.group(1))

            # 枠番
            br_m = re.match(r'^(\d+)$', tx[1].strip()) if len(tx) > 1 else None
            bracket = int(br_m.group(1)) if br_m else None

            # 馬番
            num_m = re.match(r'^(\d+)$', tx[2].strip())
            num   = int(num_m.group(1)) if num_m else 0

            # 馬名
            name_m = re.match(
                r'^([\u30A0-\u30FF\u4e00-\u9fffA-Za-z][\u30A0-\u30FF\u4e00-\u9fffA-Za-z0-9・]{1,20})',
                tx[3].strip())
            name = name_m.group(1).strip() if name_m else tx[3].strip()[:10]
            if not name or len(name) < 2: continue

            # 性齢（牡3 / 牝4 / セ5）
            sex, age = '', None
            if len(tx) > 4:
                sm = re.match(r'^([牡牝騸セ騙])\s*(\d+)', tx[4].strip())
                if sm:
                    sex = sm.group(1)
                    if sex == '騙': sex = 'セ'
                    age = int(sm.group(2))

            # 斤量
            weight_load = None
            if len(tx) > 5:
                wm = re.match(r'^(\d{2}(?:\.\d)?)$', tx[5].strip())
                if wm:
                    v = float(wm.group(1))
                    if 45.0 <= v <= 65.0: weight_load = v

            jockey  = tx[6].strip() if len(tx) > 6 else ''

            # タイム
            finish_time = _parse_finish_time(tx[7].strip() if len(tx) > 7 else '')

            # 着差
            chakusa_text = tx[8].strip() if len(tx) > 8 else ''
            margin = _parse_margin(chakusa_text)

            # 通過順
            corner_all_text = tx[9] if len(tx) > 9 else ''
            pos_nums = re.findall(r'\d+', corner_all_text)
            if pos_nums:
                positions = [int(n) for n in pos_nums[:4]]
                first = positions[0]; avg = sum(positions)/len(positions)
                style = ('逃げ' if first==1 else '先行' if avg<=3 else '差し' if avg<=7 else '追込')
                corner_3 = positions[-2] if len(positions) >= 2 else positions[0]
                corner_4 = positions[-1]
                corner_all = '-'.join(pos_nums[:4])
            else:
                style = '差し'; corner_3 = corner_4 = None; corner_all = ''

            # 上がり3F
            agari_m = re.search(r'(\d{2}\.\d)', tx[10]) if len(tx) > 10 else None
            agari   = float(agari_m.group(1)) if agari_m else 0.0

            # 単勝オッズ（tx[11]付近）
            win_odds = None
            if len(tx) > 11:
                om = re.match(r'^(\d{1,4}\.\d)$', tx[11].strip())
                if om:
                    v = float(om.group(1))
                    if 1.0 <= v <= 9999.9: win_odds = v

            # 人気
            pop_m = re.match(r'^(\d+)$', tx[12].strip()) if len(tx) > 12 else None
            popularity = int(pop_m.group(1)) if pop_m else 99

            # 馬体重
            body_weight = body_weight_diff = None
            if len(tx) > 13:
                bwm = re.match(r'^(\d{3,4})\s*[\(（]\s*([+-]?\d{1,3})\s*[\)）]', tx[13].strip())
                if bwm:
                    body_weight = int(bwm.group(1))
                    body_weight_diff = int(bwm.group(2))
                else:
                    bwm2 = re.match(r'^(\d{3,4})$', tx[13].strip())
                    if bwm2:
                        v = int(bwm2.group(1))
                        if 300 <= v <= 700: body_weight = v

            trainer = tx[14].strip() if len(tx) > 14 else ''

            finishers.append({
                'place': place, 'num': num, 'name': name,
                'running_style': style, 'agari3f': agari,
                'popularity': popularity,
                'jockey': jockey, 'trainer': trainer,
                'distance': distance, 'surface': surface, 'racecourse': rc,
                'corner_3': corner_3, 'corner_4': corner_4, 'corner_all': corner_all,
                # 新フィールド
                'bracket': bracket,
                'sex': sex, 'age': age,
                'weight_load': weight_load,
                'finish_time': finish_time,
                'chakusa_text': chakusa_text,
                'margin': margin,
                'body_weight': body_weight,
                'body_weight_diff': body_weight_diff,
                'win_odds': win_odds,
            })

        if not finishers: return None

        # agari_rank を計算
        valid = sorted([(i, h['agari3f']) for i, h in enumerate(finishers) if h['agari3f'] > 0],
                       key=lambda x: x[1])
        for rank, (i, _) in enumerate(valid):
            finishers[i]['agari_rank'] = rank + 1
        for h in finishers:
            if 'agari_rank' not in h: h['agari_rank'] = 99

        # time_diff_sec（勝ち馬との差秒）
        winner = next((h for h in finishers if h['place'] == 1), None)
        wt = winner['finish_time'] if winner and winner.get('finish_time') else None
        for h in finishers:
            ft = h.get('finish_time')
            h['time_diff_sec'] = round(ft - wt, 2) if (wt and ft) else None

        # 払戻
        text = soup.get_text(' ', strip=True)
        tm = re.search(r'単勝\s+(\d+)\s+([\d,]+)\s*円', text)
        tansho_num    = int(tm.group(1)) if tm else 0
        tansho_payout = int(tm.group(2).replace(',','')) if tm else 0
        fuku_list = []
        idx = text.find('複勝')
        if idx >= 0:
            fm = re.findall(r'(\d+)\s+([\d,]+)\s*円', text[idx:idx+200])
            fuku_list = [{'num':int(f[0]),'payout':int(f[1].replace(',',''))} for f in fm[:3]]

        return {
            'race_id': f'{date}_{pc}_{race_num:02d}',
            'date': f'{date[:4]}-{date[4:6]}-{date[6:]}',
            'racecourse': rc, 'race_num': race_num, 'race_name': rname,
            'distance': distance, 'surface': surface,
            'lap_times': lap_times, 'first_3f': first_3f, 'last_3f': last_3f,
            'track_condition': track_condition,
            'race_class': race_class,
            'weather': weather,
            'num_finishers': len(finishers),
            'finishers': finishers,
            'tansho_num': tansho_num, 'tansho_payout': tansho_payout,
            'fukusho': fuku_list,
        }
    except Exception as e:
        return None


def update_race_history(result, conn):
    """race_history を UPDATE（INSERT OR IGNOREで新規追加 + UPDATE で全列補完）"""
    # 新規の場合は INSERT
    conn.execute(
        "INSERT OR IGNORE INTO race_history "
        "(race_id,date,racecourse,race_num,race_name,distance,surface,"
        " lap_times,first_3f,last_3f,track_condition,race_class,num_finishers,weather,pace_label) "
        "VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
        (result['race_id'], result['date'], result['racecourse'], result['race_num'],
         result['race_name'], result['distance'], result['surface'],
         result.get('lap_times'), result.get('first_3f'), result.get('last_3f'),
         result.get('track_condition'), result.get('race_class'),
         result.get('num_finishers'), result.get('weather'), None)
    )
    # 既存・新規とも UPDATE で全フィールドを補完
    conn.execute(
        """UPDATE race_history SET
            date            = ?,
            surface         = CASE WHEN ? != '不明' THEN ? ELSE surface END,
            race_name       = COALESCE(NULLIF(?, ''), race_name),
            track_condition = COALESCE(?, track_condition),
            race_class      = COALESCE(NULLIF(?, ''), race_class),
            num_finishers   = COALESCE(?, num_finishers),
            weather         = COALESCE(?, weather),
            lap_times       = COALESCE(NULLIF(?, ''), lap_times),
            first_3f        = COALESCE(?, first_3f),
            last_3f         = COALESCE(?, last_3f)
        WHERE race_id = ?""",
        (result['date'],
         result.get('surface','不明'), result.get('surface'),
         result.get('race_name',''),
         result.get('track_condition'),
         result.get('race_class',''),
         result.get('num_finishers'),
         result.get('weather'),
         result.get('lap_times',''),
         result.get('first_3f'),
         result.get('last_3f'),
         result['race_id'])
    )

    for h in result.get('finishers', []):
        fuku = next((f['payout'] for f in result['fukusho'] if f['num'] == h['num']), 0)
        tan  = result['tansho_payout'] if h['num'] == result['tansho_num'] else 0
        # 新規馬は INSERT
        conn.execute(
            "INSERT OR IGNORE INTO horse_history "
            "(race_id,date,racecourse,race_name,distance,surface,place,horse_num,horse_name,"
            " running_style,agari3f,popularity,jockey,trainer,tansho_payout,fukusho_payout,"
            " corner_3,corner_4) VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
            (result['race_id'], result['date'], result['racecourse'], result['race_name'],
             result['distance'], result['surface'],
             h['place'], h['num'], h['name'], h['running_style'], h['agari3f'],
             h['popularity'], h.get('jockey',''), h.get('trainer',''), tan, fuku,
             h.get('corner_3'), h.get('corner_4'))
        )
        # 既存・新規とも UPDATE で新フィールドを補完
        conn.execute(
            """UPDATE horse_history SET
                surface          = CASE WHEN ? != '不明' THEN ? ELSE surface END,
                weight_load      = COALESCE(?, weight_load),
                sex              = COALESCE(NULLIF(?, ''), sex),
                age              = COALESCE(?, age),
                body_weight      = COALESCE(?, body_weight),
                body_weight_diff = COALESCE(?, body_weight_diff),
                bracket          = COALESCE(?, bracket),
                corner_all       = COALESCE(NULLIF(?, ''), corner_all),
                win_odds         = COALESCE(?, win_odds),
                finish_time      = COALESCE(?, finish_time),
                time_diff_sec    = COALESCE(?, time_diff_sec),
                chakusa_text     = COALESCE(NULLIF(?, ''), chakusa_text),
                margin           = COALESCE(?, margin),
                agari_rank       = COALESCE(?, agari_rank),
                corner_3         = COALESCE(?, corner_3),
                corner_4         = COALESCE(?, corner_4),
                agari3f          = COALESCE(NULLIF(?, 0.0), agari3f)
            WHERE race_id = ? AND horse_num = ?""",
            (h.get('surface','不明'), h.get('surface'),
             h.get('weight_load'), h.get('sex',''), h.get('age'),
             h.get('body_weight'), h.get('body_weight_diff'),
             h.get('bracket'), h.get('corner_all',''),
             h.get('win_odds'), h.get('finish_time'), h.get('time_diff_sec'),
             h.get('chakusa_text',''), h.get('margin'), h.get('agari_rank'),
             h.get('corner_3'), h.get('corner_4'), h.get('agari3f'),
             result['race_id'], h['num'])
        )

print('✅ 関数定義完了')
conn_check = sqlite3.connect(HIST_DB)
rc = conn_check.execute('SELECT COUNT(*) FROM race_history').fetchone()[0]
hc = conn_check.execute('SELECT COUNT(*) FROM horse_history').fetchone()[0]
conn_check.close()
print(f'📊 DB: {rc:,}レース / {hc:,}頭')

In [ ]:
# ── セル5: マイグレーション（新列追加・冪等）────────────────
conn = sqlite3.connect(HIST_DB)
horse_new = [
    ('weight_load','REAL'),('sex','TEXT'),('age','INTEGER'),
    ('body_weight','INTEGER'),('body_weight_diff','INTEGER'),
    ('bracket','INTEGER'),('corner_all','TEXT'),('win_odds','REAL'),
    ('margin','REAL'),('agari_rank','INTEGER'),('finish_time','REAL'),
    ('time_diff_sec','REAL'),('chakusa_text','TEXT'),
]
race_new = [
    ('track_condition','TEXT'),('race_class','TEXT'),('num_finishers','INTEGER'),
    ('weather','TEXT'),('pace_label','TEXT'),('lap_times','TEXT'),
    ('first_3f','REAL'),('last_3f','REAL'),
]
existing_h = {r[1] for r in conn.execute('PRAGMA table_info(horse_history)')}
for col, typ in horse_new:
    if col not in existing_h:
        conn.execute(f'ALTER TABLE horse_history ADD COLUMN {col} {typ}')
        print(f'  ➕ horse_history.{col}')
existing_r = {r[1] for r in conn.execute('PRAGMA table_info(race_history)')}
for col, typ in race_new:
    if col not in existing_r:
        conn.execute(f'ALTER TABLE race_history ADD COLUMN {col} {typ}')
        print(f'  ➕ race_history.{col}')
conn.commit(); conn.close()
print('✅ マイグレーション完了')

In [ ]:
# ── セル6: セッション準備 + suffix_map 読み込み ──────────────
sess = requests.Session()
sess.headers.update(HEADERS)
retry = Retry(total=3, backoff_factor=2, status_forcelist=[500,502,503,504])
sess.mount('https://', HTTPAdapter(max_retries=retry))
sess.get(f'{JRA_BASE}/keiba/thisweek/', timeout=15)

suffix_path = f'{BASE_DIR}/data/month_suffix_map.json'
if os.path.exists(suffix_path):
    with open(suffix_path) as f:
        suffix_map = json.load(f)
    print(f'✅ suffix_map: {len(suffix_map)}ヶ月分')
else:
    print('❌ month_suffix_map.json が見つかりません')
    print('   → 一括取得v5のセル3bを実行してから再試行してください')
    suffix_map = {}

In [ ]:
# ── セル7: ★再スクレイプ本体 ─────────────────────────────────
# v5と同じURL構造（sde_base = pw01sde10{pc}{year}{kai}{nichi}）を使用
# 既存レースも全フィールドをUPDATEで補完
# ★ランタイム切れ後の再開: セル1〜6を実行してからこのセルを実行するだけでOK
#   → 既に track_condition が埋まっている開催日は自動スキップする

START_YM = '202501'  # ★ 取得開始月
END_YM   = '202605'  # ★ 取得終了月（直前週まで）

# ── 完了済み「date_pc」をDBから取得（再開用） ──
conn_chk = sqlite3.connect(HIST_DB)
done_rows = conn_chk.execute(
    "SELECT SUBSTR(REPLACE(date,'-',''), 1, 8), "
    "       SUBSTR(race_id, 10, 2) "
    "FROM race_history "
    "WHERE track_condition IS NOT NULL "
    "GROUP BY 1, 2"
).fetchall()
conn_chk.close()
done_set = {(d, pc) for d, pc in done_rows}
if done_set:
    sample = sorted(done_set)[-1]
    print(f'✅ 完了済み: {len(done_set)}件の開催日×競馬場（最終: {sample}）→ スキップします')
else:
    print('📋 完了済み: なし（最初から実行）')

def month_range(start, end):
    months = []
    y, m = int(start[:4]), int(start[4:])
    ey, em = int(end[:4]), int(end[4:])
    while (y, m) <= (ey, em):
        months.append(f'{y:04d}{m:02d}')
        m += 1
        if m > 12: m, y = 1, y+1
    return months

target_months = [ym for ym in month_range(START_YM, END_YM) if ym in suffix_map]
missing = [ym for ym in month_range(START_YM, END_YM) if ym not in suffix_map]
if missing: print(f'⚠ suffix未取得: {missing}')
print(f'対象: {len(target_months)}ヶ月 ({target_months[0]}〜{target_months[-1]})')

total_updated = 0
errors = []
start_time = time.time()

for ym in target_months:
    print(f'\n{"="*40}')
    print(f'📅 {ym[:4]}年{ym[4:]}月...')

    links = get_kaisai_for_month(ym, suffix_map[ym], sess)
    if not links:
        print('  開催なし'); continue

    by_date = defaultdict(list)
    for base, date in sorted(links.items(), key=lambda x: x[1]):
        by_date[date].append(base)

    for date_str in sorted(by_date.keys()):
        print(f'\n  [{date_str}]')
        for sde_base in by_date[date_str]:
            pc      = sde_base[9:11]
            rc_name = PLACE_NAMES.get(pc, '?')

            # ── 再開スキップ ──
            if (date_str, pc) in done_set:
                print(f'    {rc_name}: ✅ 完了済みスキップ')
                continue

            try:
                r01 = find_r01_result(sde_base, date_str, sess)
                if r01 is None:
                    print(f'    {rc_name}: ⚠ suffix取得失敗')
                    errors.append(f'{date_str} {rc_name}')
                    continue

                saved = 0
                conn = sqlite3.connect(HIST_DB)
                conn.execute('PRAGMA journal_mode=WAL')

                for r in range(1, 13):
                    sx  = calc_suffix(r01, r)
                    cn  = f'{sde_base}{r:02d}{date_str}/{sx}'
                    try:
                        resp = sess.post(f'{JRA_BASE}/JRADB/accessS.html',
                                         data={'CNAME': cn}, timeout=30)
                        resp.encoding = 'shift_jis'
                        if 'パラメータエラー' in resp.text: continue
                        soup = BeautifulSoup(resp.text, 'lxml')
                        if not soup.find_all('table'): continue
                        result = parse_result_page(soup, rc_name, r, date_str, pc)
                        if not result: continue
                        update_race_history(result, conn)
                        saved += 1
                        total_updated += 1
                        time.sleep(0.8)
                    except Exception as e:
                        print(f'    {rc_name} R{r}: ⚠ {type(e).__name__}')
                        time.sleep(2)

                conn.commit(); conn.close()
                elapsed = time.time() - start_time
                print(f'    {rc_name}: {saved}レース更新  累計{total_updated}  経過{elapsed/60:.1f}分')

            except Exception as e:
                print(f'    {rc_name}: ❌ {e}')
                errors.append(f'{date_str} {rc_name}: {e}')
                time.sleep(3)

elapsed = time.time() - start_time
print(f'\n✅ 完了: {total_updated}レース更新  所要{elapsed/60:.1f}分')
if errors:
    print(f'⚠ 失敗: {len(errors)}件')
    for e in errors[:10]: print(f'  {e}')

In [ ]:
# ── セル8: 充足率検証 ─────────────────────────────────────────
conn = sqlite3.connect(HIST_DB)

print('=== race_history ===')
total_r = conn.execute('SELECT COUNT(*) FROM race_history').fetchone()[0]
for col in ['surface','track_condition','race_class','weather','num_finishers','race_name']:
    filled = conn.execute(
        f"SELECT COUNT(*) FROM race_history WHERE {col} IS NOT NULL AND {col} != ''"
    ).fetchone()[0]
    pct = 100 * filled / max(total_r, 1)
    mark = '✅' if pct >= 90 else '⚠' if pct >= 70 else '❌'
    print(f'  {mark} {col}: {filled}/{total_r} ({pct:.1f}%)')

print('\n=== horse_history ===')
total_h = conn.execute('SELECT COUNT(*) FROM horse_history').fetchone()[0]
for col in ['surface','weight_load','sex','age','body_weight','bracket','corner_all','win_odds','finish_time']:
    filled = conn.execute(
        f"SELECT COUNT(*) FROM horse_history WHERE {col} IS NOT NULL AND {col} != ''"
    ).fetchone()[0]
    pct = 100 * filled / max(total_h, 1)
    mark = '✅' if pct >= 90 else '⚠' if pct >= 70 else '❌'
    print(f'  {mark} {col}: {filled}/{total_h} ({pct:.1f}%)')

print('\n=== surface分布（芝:ダート ≈ 55:45 が正常）===')
for tbl in ['race_history','horse_history']:
    rows = conn.execute(
        f'SELECT surface, COUNT(*) FROM {tbl} GROUP BY surface ORDER BY 2 DESC'
    ).fetchall()
    print(f'  {tbl}: {dict(rows)}')

conn.close()